In [82]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [83]:
import os

os.listdir('/content/drive/MyDrive/UPI_Fraud_Project')

['indian_banking_transactions.csv']

In [84]:
import pandas as pd

file_path = '/content/drive/MyDrive/UPI_Fraud_Project/indian_banking_transactions.csv'

df = pd.read_csv(file_path)

print(df.shape)

(550000, 20)


In [85]:
df.head()

,transaction_id,customer_id,transaction_date,transaction_time,account_type,transaction_type,transaction_amount,transaction_direction,account_balance,merchant_category,state,credit_score,has_loan,loan_type,emi_amount,transaction_status,channel,kyc_status,is_fraud,transaction_hour
0,TXN000000001,CUST015796,2019-01-01,15:28,Current,UPI,1820.17,Debit,609365.31,Food & Dining,Maharashtra,764,0,NaN,0.00,Success,Branch,Verified,0,15
1,TXN000000002,CUST000861,2019-01-01,03:00,Current,UPI,392.67,Credit,14451.14,Education,West Bengal,630,0,NaN,0.00,Success,Mobile_App,Verified,0,3
2,TXN000000003,CUST076821,2019-01-01,18:03,Fixed Deposit,POS,1255.97,Debit,47621.87,Utilities,Punjab,813,1,Home,1433.91,Success,Mobile_App,Verified,0,18
3,TXN000000004,CUST054887,2019-01-01,08:03,Savings,UPI,2580.68,Debit,34467.85,Travel,Karnataka,628,1,Personal,6280.35,Success,API,Verified,0,8
4,TXN000000005,CUST006266,2019-01-01,14:23,Fixed Deposit,UPI,2573.80,Debit,26617.40,Utilities,West Bengal,767,0,NaN,0.00,Success,API,Verified,0,14


In [86]:
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())

(550000, 20)
['transaction_id', 'customer_id', 'transaction_date', 'transaction_time', 'account_type', 'transaction_type', 'transaction_amount', 'transaction_direction', 'account_balance', 'merchant_category', 'state', 'credit_score', 'has_loan', 'loan_type', 'emi_amount', 'transaction_status', 'channel', 'kyc_status', 'is_fraud', 'transaction_hour']
transaction_id                0
customer_id                   0
transaction_date              0
transaction_time              0
account_type                  0
transaction_type              0
transaction_amount            0
transaction_direction         0
account_balance               0
merchant_category             0
state                         0
credit_score                  0
has_loan                      0
loan_type                377134
emi_amount                    0
transaction_status            0
channel                       0
kyc_status                    0
is_fraud                      0
transaction_hour              0
dtype: 

In [87]:
print(df['is_fraud'].value_counts())
print(df['is_fraud'].value_counts(normalize=True) * 100)

is_fraud
0    545127
1      4873
Name: count, dtype: int64
is_fraud
0    99.114
1     0.886
Name: proportion, dtype: float64


In [88]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

df['transaction_year'] = df['transaction_date'].dt.year
df['transaction_month'] = df['transaction_date'].dt.month
df['transaction_day'] = df['transaction_date'].dt.day
df['transaction_dayofweek'] = df['transaction_date'].dt.dayofweek

print(df[['transaction_date',
          'transaction_year',
          'transaction_month',
          'transaction_day',
          'transaction_dayofweek']].head())

  transaction_date  transaction_year  transaction_month  transaction_day  \
0       2019-01-01              2019                  1                1   
1       2019-01-01              2019                  1                1   
2       2019-01-01              2019                  1                1   
3       2019-01-01              2019                  1                1   
4       2019-01-01              2019                  1                1   

   transaction_dayofweek  
0                      1  
1                      1  
2                      1  
3                      1  
4                      1  


In [89]:
print(df[['transaction_date',
          'transaction_year',
          'transaction_month',
          'transaction_day',
          'transaction_dayofweek']].iloc[[0, 100, 1000, 10000, 100000]])

       transaction_date  transaction_year  transaction_month  transaction_day  \
0            2019-01-01              2019                  1                1   
100          2019-01-01              2019                  1                1   
1000         2019-01-04              2019                  1                4   
10000        2019-02-03              2019                  2                3   
100000       2019-11-28              2019                 11               28   

        transaction_dayofweek  
0                           1  
100                         1  
1000                        4  
10000                       6  
100000                      3  


In [90]:
df['amount_balance_ratio'] = (
    df['transaction_amount'] / (df['account_balance'] + 1)
)

print(df[['transaction_amount',
          'account_balance',
          'amount_balance_ratio']].head())

   transaction_amount  account_balance  amount_balance_ratio
0             1820.17        609365.31              0.002987
1              392.67         14451.14              0.027170
2             1255.97         47621.87              0.026373
3             2580.68         34467.85              0.074870
4             2573.80         26617.40              0.096693


In [91]:
df['balance_after_transaction'] = (
    df['account_balance'] - df['transaction_amount']
)

print(df[['account_balance',
          'transaction_amount',
          'balance_after_transaction']].head())

   account_balance  transaction_amount  balance_after_transaction
0        609365.31             1820.17                  607545.14
1         14451.14              392.67                   14058.47
2         47621.87             1255.97                   46365.90
3         34467.85             2580.68                   31887.17
4         26617.40             2573.80                   24043.60


In [92]:
high_value_threshold = df['transaction_amount'].quantile(0.95)

df['high_value_transaction'] = (
    df['transaction_amount'] > high_value_threshold
).astype(int)

print("High-value threshold:", high_value_threshold)
print(df['high_value_transaction'].value_counts())

High-value threshold: 140856.61599999998
high_value_transaction
0    522500
1     27500
Name: count, dtype: int64


In [93]:
df['night_transaction'] = (
    (df['transaction_hour'] >= 22) |
    (df['transaction_hour'] <= 5)
).astype(int)

print(df['night_transaction'].value_counts())

night_transaction
0    366709
1    183291
Name: count, dtype: int64


In [94]:
df['weekend_transaction'] = (
    df['transaction_dayofweek'] >= 5
).astype(int)

print(df['weekend_transaction'].value_counts())

weekend_transaction
0    392606
1    157394
Name: count, dtype: int64


In [95]:
print("Total columns:", len(df.columns))
print(df.columns.tolist())

Total columns: 29
['transaction_id', 'customer_id', 'transaction_date', 'transaction_time', 'account_type', 'transaction_type', 'transaction_amount', 'transaction_direction', 'account_balance', 'merchant_category', 'state', 'credit_score', 'has_loan', 'loan_type', 'emi_amount', 'transaction_status', 'channel', 'kyc_status', 'is_fraud', 'transaction_hour', 'transaction_year', 'transaction_month', 'transaction_day', 'transaction_dayofweek', 'amount_balance_ratio', 'balance_after_transaction', 'high_value_transaction', 'night_transaction', 'weekend_transaction']


In [96]:
# Behavioral feature engineering

In [97]:
# Sort transactions by customer and date/time
df['transaction_datetime'] = pd.to_datetime(
    df['transaction_date'].astype(str) + ' ' + df['transaction_time'].astype(str)
)

df = df.sort_values(['customer_id', 'transaction_datetime']).reset_index(drop=True)

print("Transactions sorted by customer and time.")

Transactions sorted by customer and time.


In [98]:
# Calculate each customer's previous average transaction amount
df['customer_avg_amount'] = (
    df.groupby('customer_id')['transaction_amount']
      .transform(lambda x: x.shift().expanding().mean())
)

# For the customer's first transaction, no previous history exists
df['customer_avg_amount'] = df['customer_avg_amount'].fillna(
    df['transaction_amount']
)

print("Behavioral feature created: customer_avg_amount")
print(df[['customer_id', 'transaction_amount', 'customer_avg_amount']].head(10))

Behavioral feature created: customer_avg_amount
  customer_id  transaction_amount  customer_avg_amount
0  CUST000001              892.86           892.860000
1  CUST000001              328.91           892.860000
2  CUST000001             1000.00           610.885000
3  CUST000001              584.18           740.590000
4  CUST000001              470.63           701.487500
5  CUST000002            65982.76         65982.760000
6  CUST000002              186.60         65982.760000
7  CUST000002            14880.67         33084.680000
8  CUST000002              384.28         27016.676667
9  CUST000002             1783.51         20358.577500


In [99]:
# Calculate how much the current transaction differs from the customer's normal amount
df['amount_deviation'] = (
    df['transaction_amount'] / (df['customer_avg_amount'] + 1)
)

print("Behavioral feature created: amount_deviation")
print(df[['customer_id',
          'transaction_amount',
          'customer_avg_amount',
          'amount_deviation']].head(10))

Behavioral feature created: amount_deviation
  customer_id  transaction_amount  customer_avg_amount  amount_deviation
0  CUST000001              892.86           892.860000          0.998881
1  CUST000001              328.91           892.860000          0.367966
2  CUST000001             1000.00           610.885000          1.634294
3  CUST000001              584.18           740.590000          0.787740
4  CUST000001              470.63           701.487500          0.669948
5  CUST000002            65982.76         65982.760000          0.999985
6  CUST000002              186.60         65982.760000          0.002828
7  CUST000002            14880.67         33084.680000          0.449762
8  CUST000002              384.28         27016.676667          0.014223
9  CUST000002             1783.51         20358.577500          0.087601


In [100]:
# Count previous transactions made by each customer
df['customer_transaction_count'] = (
    df.groupby('customer_id').cumcount()
)

print("Behavioral feature created: customer_transaction_count")
print(df[['customer_id',
          'transaction_amount',
          'customer_transaction_count']].head(10))

Behavioral feature created: customer_transaction_count
  customer_id  transaction_amount  customer_transaction_count
0  CUST000001              892.86                           0
1  CUST000001              328.91                           1
2  CUST000001             1000.00                           2
3  CUST000001              584.18                           3
4  CUST000001              470.63                           4
5  CUST000002            65982.76                           0
6  CUST000002              186.60                           1
7  CUST000002            14880.67                           2
8  CUST000002              384.28                           3
9  CUST000002             1783.51                           4


In [101]:
# Next cell — verify all 3 together

In [102]:
behavioral_features = [
    'customer_avg_amount',
    'amount_deviation',
    'customer_transaction_count'
]

print("Behavioral features created:")
print(behavioral_features)

print("\nSample:")
print(df[
    ['customer_id',
     'transaction_amount',
     'customer_avg_amount',
     'amount_deviation',
     'customer_transaction_count']
].head(10))

Behavioral features created:
['customer_avg_amount', 'amount_deviation', 'customer_transaction_count']

Sample:
  customer_id  transaction_amount  customer_avg_amount  amount_deviation  \
0  CUST000001              892.86           892.860000          0.998881   
1  CUST000001              328.91           892.860000          0.367966   
2  CUST000001             1000.00           610.885000          1.634294   
3  CUST000001              584.18           740.590000          0.787740   
4  CUST000001              470.63           701.487500          0.669948   
5  CUST000002            65982.76         65982.760000          0.999985   
6  CUST000002              186.60         65982.760000          0.002828   
7  CUST000002            14880.67         33084.680000          0.449762   
8  CUST000002              384.28         27016.676667          0.014223   
9  CUST000002             1783.51         20358.577500          0.087601   

   customer_transaction_count  
0                  

In [103]:
# Next cell — compare behavioral features

In [104]:
print("Behavioral feature comparison:\n")

print("Customer average amount:")
print(df.groupby('is_fraud')['customer_avg_amount'].mean())

print("\nAmount deviation:")
print(df.groupby('is_fraud')['amount_deviation'].mean())

print("\nPrevious transaction count:")
print(df.groupby('is_fraud')['customer_transaction_count'].mean())

Behavioral feature comparison:

Customer average amount:
is_fraud
0    29693.537189
1    43127.966296
Name: customer_avg_amount, dtype: float64

Amount deviation:
is_fraud
0    12.011096
1    38.142151
Name: amount_deviation, dtype: float64

Previous transaction count:
is_fraud
0    3.433681
1    3.481633
Name: customer_transaction_count, dtype: float64


In [128]:
X = df.drop(columns=['is_fraud', 'transaction_datetime'])
y = df['is_fraud']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (550000, 31)
y shape: (550000,)


In [129]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (440000, 31)
X_test shape: (110000, 31)
y_train shape: (440000,)
y_test shape: (110000,)


In [150]:
numeric_features = [
    'transaction_amount',
    'account_balance',
    'credit_score',
    'has_loan',
    'emi_amount',
    'transaction_hour',
    'transaction_year',
    'transaction_month',
    'transaction_day',
    'transaction_dayofweek',
    'amount_balance_ratio',
    'balance_after_transaction',
    'high_value_transaction',
    'night_transaction',
    'weekend_transaction',
    'customer_avg_amount',
    'amount_deviation',
    'customer_transaction_count'
]

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numeric_features))

Categorical features: 9
Numerical features: 18


In [153]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print("Preprocessor updated.")
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Preprocessor updated.
Numerical features: 18
Categorical features: 9


In [154]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

Processed X_train shape: (440000, 79)
Processed X_test shape: (110000, 79)


In [155]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_processed, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [156]:
y_pred_lr = lr_model.predict(X_test_processed)

print("Predictions generated successfully.")

Predictions generated successfully.


In [157]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))

y_prob_lr = lr_model.predict_proba(X_test_processed)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))
print("PR-AUC:", average_precision_score(y_test, y_prob_lr))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

Accuracy: 0.7558545454545454
Precision: 0.01103344039297185
Recall: 0.2994871794871795
F1 Score: 0.02128279883381924
ROC-AUC: 0.5371247196890856
PR-AUC: 0.02085502191850197

Confusion Matrix:
[[82852 26173]
 [  683   292]]


In [158]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=30,
    max_depth=10,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train_processed, y_train)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [114]:
### Make Random Forest predictions

In [159]:
y_pred_rf = rf_model.predict(X_test_processed)

print("Random Forest predictions generated successfully.")

Random Forest predictions generated successfully.


In [116]:
#Perfect ✅ Now calculate the Random Forest metrics.

In [160]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))

y_prob_rf = rf_model.predict_proba(X_test_processed)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print("PR-AUC:", average_precision_score(y_test, y_prob_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Accuracy: 0.9673454545454545
Precision: 0.0358283079106066
Recall: 0.10358974358974359
F1 Score: 0.05324196099103848
ROC-AUC: 0.5371358204128669
PR-AUC: 0.017105750274788275

Confusion Matrix:
[[106307   2718]
 [   874    101]]


In [161]:
from xgboost import XGBClassifier

fraud_count = y_train.sum()
genuine_count = (y_train == 0).sum()

scale_pos_weight = genuine_count / fraud_count

print("scale_pos_weight:", scale_pos_weight)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_processed, y_train)

print("XGBoost model trained successfully.")

scale_pos_weight: 111.8783991790662
XGBoost model trained successfully.


In [162]:
y_pred_xgb = xgb_model.predict(X_test_processed)

print("XGBoost predictions generated successfully.")

XGBoost predictions generated successfully.


In [163]:
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))

y_prob_xgb = xgb_model.predict_proba(X_test_processed)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))
print("PR-AUC:", average_precision_score(y_test, y_prob_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

Accuracy: 0.8810363636363636
Precision: 0.014355601892693881
Recall: 0.1835897435897436
F1 Score: 0.026628979470395716
ROC-AUC: 0.5462574826992161
PR-AUC: 0.01668702133856951

Confusion Matrix:
[[96735 12290]
 [  796   179]]


In [164]:
comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Random Forest',
        'XGBoost'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    'F1 Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_xgb)
    ],
    'PR-AUC': [
        average_precision_score(y_test, y_prob_lr),
        average_precision_score(y_test, y_prob_rf),
        average_precision_score(y_test, y_prob_xgb)
    ]
})

print(comparison.round(4))

                 Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC  PR-AUC
0  Logistic Regression    0.7559     0.0110  0.2995    0.0213   0.5371  0.0209
1        Random Forest    0.9673     0.0358  0.1036    0.0532   0.5371  0.0171
2              XGBoost    0.8810     0.0144  0.1836    0.0266   0.5463  0.0167


In [165]:
comparison_percent = comparison.copy()

for col in ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'PR-AUC']:
    comparison_percent[col] = (
        comparison_percent[col] * 100
    ).round(2)

print(comparison_percent.to_string(index=False))

              Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC  PR-AUC
Logistic Regression     75.59       1.10   29.95      2.13    53.71    2.09
      Random Forest     96.73       3.58   10.36      5.32    53.71    1.71
            XGBoost     88.10       1.44   18.36      2.66    54.63    1.67


In [166]:
amount_summary = df.groupby('is_fraud')['transaction_amount'].agg(
    ['mean', 'median', 'max']
)

print(amount_summary)

                   mean   median          max
is_fraud                                     
0          29106.685733  2030.99  10000000.00
1         119446.211798  2691.95   6751663.36


In [167]:
type_analysis = df.groupby('transaction_type')['is_fraud'].agg(
    ['count', 'sum', 'mean']
)

type_analysis['fraud_rate_percent'] = type_analysis['mean'] * 100

print(
    type_analysis
    .sort_values('fraud_rate_percent', ascending=False)
    .round(4)
)

                   count   sum    mean  fraud_rate_percent
transaction_type                                          
RTGS               32964   611  0.0185              1.8535
Cheque             21711   256  0.0118              1.1791
NEFT               66615   581  0.0087              0.8722
Credit_Card        16426   140  0.0085              0.8523
Net_Banking        38458   326  0.0085              0.8477
ATM_Withdrawal     55106   456  0.0083              0.8275
IMPS               77038   624  0.0081              0.8100
POS                60278   481  0.0080              0.7980
UPI               153986  1211  0.0079              0.7864
Auto_Debit         27418   187  0.0068              0.6820


In [168]:
high_value_analysis = df.groupby('high_value_transaction')['is_fraud'].agg(
    ['count', 'sum', 'mean']
)

high_value_analysis['fraud_rate_percent'] = (
    high_value_analysis['mean'] * 100
)

print(high_value_analysis.round(4))

                         count   sum   mean  fraud_rate_percent
high_value_transaction                                         
0                       522500  4185  0.008              0.8010
1                        27500   688  0.025              2.5018


In [169]:
ratio_analysis = df.groupby('is_fraud')['amount_balance_ratio'].agg(
    ['mean', 'median', 'max']
)

print(ratio_analysis)

              mean    median          max
is_fraud                                 
0         1.868906  0.065420  6772.161119
1         6.919420  0.089832  2116.343684


In [170]:
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df.head(15))

                            feature  importance
11   num__balance_after_transaction    0.166251
0           num__transaction_amount    0.155496
10        num__amount_balance_ratio    0.080695
16            num__amount_deviation    0.056412
15         num__customer_avg_amount    0.043928
1              num__account_balance    0.042014
2                 num__credit_score    0.041791
12      num__high_value_transaction    0.037384
8              num__transaction_day    0.028078
4                   num__emi_amount    0.027363
5             num__transaction_hour    0.026928
7            num__transaction_month    0.025331
17  num__customer_transaction_count    0.022926
31       cat__transaction_type_RTGS    0.018304
9        num__transaction_dayofweek    0.017222


In [171]:
print("Behavioral Feature Analysis\n")

print("1. Amount Deviation:")
print(df.groupby('is_fraud')['amount_deviation'].mean())

print("\n2. Customer Average Amount:")
print(df.groupby('is_fraud')['customer_avg_amount'].mean())

print("\n3. Customer Transaction Count:")
print(df.groupby('is_fraud')['customer_transaction_count'].mean())

Behavioral Feature Analysis

1. Amount Deviation:
is_fraud
0    12.011096
1    38.142151
Name: amount_deviation, dtype: float64

2. Customer Average Amount:
is_fraud
0    29693.537189
1    43127.966296
Name: customer_avg_amount, dtype: float64

3. Customer Transaction Count:
is_fraud
0    3.433681
1    3.481633
Name: customer_transaction_count, dtype: float64


In [172]:
# Generate fraud probability scores using Random Forest

fraud_score = rf_model.predict_proba(X_test_processed)[:, 1]

print("Sample fraud scores:")
print(fraud_score[:10])

Sample fraud scores:
[0.45306999 0.48831413 0.44930926 0.48094103 0.42613256 0.30581703
 0.45998037 0.42889948 0.45380013 0.41365984]


In [173]:
# Convert fraud score into binary fraud prediction

threshold = 0.5

fraud_prediction = (fraud_score >= threshold).astype(int)

print("Threshold:", threshold)
print("Sample fraud scores:", fraud_score[:10])
print("Sample fraud predictions:", fraud_prediction[:10])

Threshold: 0.5
Sample fraud scores: [0.45306999 0.48831413 0.44930926 0.48094103 0.42613256 0.30581703
 0.45998037 0.42889948 0.45380013 0.41365984]
Sample fraud predictions: [0 0 0 0 0 0 0 0 0 0]


In [ ]:
# higher fraud score actually results in a fraud flag.

In [174]:
# Show transactions with the highest fraud scores

top_scores = pd.DataFrame({
    'Fraud Score': fraud_score,
    'Prediction': fraud_prediction
}).sort_values('Fraud Score', ascending=False)

print(top_scores.head(10))

        Fraud Score  Prediction
21324      0.909803           1
28195      0.907075           1
15414      0.904633           1
103971     0.901135           1
20605      0.900763           1
104560     0.899492           1
55107      0.898366           1
2344       0.897856           1
109883     0.897092           1
82439      0.896078           1
